# Direction Policy Kaggle Pipeline

This notebook prepares raw M5 forex CSV data, generates direction labels, and trains/replays one or more of the five neural architectures.

The first code cell now works whether Kaggle leaves your project as a `.zip` file or automatically extracts it into an input dataset folder.

Edit `RAW_DATASET_DIR`, and optionally set `PROJECT_INPUT` to your Kaggle project dataset folder if auto-detection does not find it.


In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, zipfile

# === EDIT THESE ===
# This can point to either:
#   1) a Kaggle input folder containing the unzipped project, e.g. /kaggle/input/direction-policy-kaggle-ready
#   2) a direct zip file, e.g. /kaggle/input/direction-policy-kaggle-ready/direction_policy_kaggle_ready.zip
#   3) None, in which case the notebook scans /kaggle/input and tries to find the project automatically.
PROJECT_INPUT = None  # e.g. '/kaggle/input/direction-policy-kaggle-ready'
RAW_DATASET_DIR = '/kaggle/input/YOUR_RAW_FOREX_DATASET'
SYMBOLS = ['EURUSD']
TRAIN_START = '2024-01-01'
TRAIN_END = '2025-03-01'
REPLAY_START = '2025-03-01'
REPLAY_END = '2025-06-01'
EPOCHS = 50
BATCH_SIZE = 512
DEVICE = 'cuda'  # use 'cpu' if no GPU accelerator is enabled
RESET_WORKDIR = True

WORKDIR = Path('/kaggle/working/project')
WORKDIR.parent.mkdir(parents=True, exist_ok=True)


def is_project_root(p: Path) -> bool:
    return (
        p.is_dir()
        and (p / 'kaggle_run_pipeline.py').exists()
        and (p / 'requirements_kaggle.txt').exists()
        and (p / 'src').is_dir()
        and (p / 'config').is_dir()
    )


def find_unzipped_project_root(base: Path) -> Path | None:
    """Find the project root when Kaggle has already unzipped the uploaded project dataset."""
    if is_project_root(base):
        return base
    if not base.exists() or not base.is_dir():
        return None

    # Search shallow first because Kaggle datasets often extract files directly under the dataset folder.
    for child in base.iterdir():
        if is_project_root(child):
            return child

    # Fallback recursive search, kept bounded enough for normal Kaggle datasets.
    for p in base.rglob('kaggle_run_pipeline.py'):
        root = p.parent
        if is_project_root(root):
            return root
    return None


def find_project_zip(base: Path) -> Path | None:
    if base.is_file() and base.suffix.lower() == '.zip':
        return base
    if base.exists() and base.is_dir():
        preferred = list(base.glob('direction_policy_kaggle_ready*.zip'))
        if preferred:
            return preferred[0]
        zips = list(base.glob('*.zip'))
        if zips:
            return zips[0]
    return None


def autodetect_project_input() -> Path:
    kaggle_input = Path('/kaggle/input')
    candidates = []
    if PROJECT_INPUT:
        candidates.append(Path(PROJECT_INPUT))
    if kaggle_input.exists():
        candidates.extend([p for p in kaggle_input.iterdir() if p.exists()])

    # Prefer an already-unzipped project root.
    for c in candidates:
        root = find_unzipped_project_root(c)
        if root is not None:
            print('Detected unzipped project root:', root)
            return root

    # Fallback to a zip file.
    for c in candidates:
        z = find_project_zip(c)
        if z is not None:
            print('Detected project zip:', z)
            return z

    checked = '\n'.join(str(c) for c in candidates)
    raise FileNotFoundError(
        'Could not find the Kaggle-ready project. Set PROJECT_INPUT to your project dataset folder or zip.\n'
        f'Checked:\n{checked}'
    )


project_source = autodetect_project_input()

if RESET_WORKDIR and WORKDIR.exists():
    shutil.rmtree(WORKDIR)

if project_source.is_file() and project_source.suffix.lower() == '.zip':
    WORKDIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(project_source, 'r') as zf:
        zf.extractall(WORKDIR)
else:
    shutil.copytree(
        project_source,
        WORKDIR,
        dirs_exist_ok=True,
        ignore=shutil.ignore_patterns('__pycache__', '*.pyc', '.git', 'models', 'logs')
    )

os.chdir(WORKDIR)
print('Project directory:', Path.cwd())
print('Raw dataset directory:', RAW_DATASET_DIR)
print('Available configs:')
for p in sorted(Path('config').glob('direction_settings*.yaml')):
    print(' -', p)


In [ ]:
# Install Kaggle-safe requirements. MetaTrader5 is intentionally not installed.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements_kaggle.txt'], check=True)


In [ ]:
# Optional smoke test: set these for a fast small run. For full training, leave as None.
RAW_MAX_ROWS_PER_SYMBOL = None      # e.g. 120000
TRAIN_MAX_ROWS = None               # e.g. 30000
MODE = 'train-all'                  # 'prepare-only', 'train-one', 'train-all', or 'grid'
CONFIGS = [
    'config/direction_settings_residual_mlp.yaml',
    'config/direction_settings_tcn.yaml',
    'config/direction_settings_inception_time.yaml',
    'config/direction_settings_small_transformer.yaml',
    'config/direction_settings_mixture_of_experts.yaml',
]

cmd = [
    sys.executable, 'kaggle_run_pipeline.py',
    '--raw-input-dir', RAW_DATASET_DIR,
    '--symbols', *SYMBOLS,
    '--timeframe', 'M5',
    '--train-start', TRAIN_START,
    '--train-end', TRAIN_END,
    '--replay-start', REPLAY_START,
    '--replay-end', REPLAY_END,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--device', DEVICE,
    '--mode', MODE,
    '--configs', *CONFIGS,
]
if RAW_MAX_ROWS_PER_SYMBOL is not None:
    cmd += ['--raw-max-rows-per-symbol', str(RAW_MAX_ROWS_PER_SYMBOL)]
if TRAIN_MAX_ROWS is not None:
    cmd += ['--train-max-rows', str(TRAIN_MAX_ROWS)]

print(' '.join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
# Package outputs for download from the Kaggle notebook output panel.
output_zip = Path('/kaggle/working/direction_policy_outputs.zip')
items = ['models', 'logs', 'data/direction', 'config/generated_spread_risk.yaml']
with zipfile.ZipFile(output_zip, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for item in items:
        p = Path(item)
        if not p.exists():
            continue
        if p.is_file():
            zf.write(p, p.as_posix())
        else:
            for f in p.rglob('*'):
                if f.is_file():
                    zf.write(f, f.as_posix())
print('Wrote', output_zip)
